In [ ]:
"""
Gene-to-m/z Spatial Pattern Matching Pipeline V12 (No Transformer)
==================================================================

KEY PARAMETERS:
- RNA (Visium): ~55 μm pixel size, hexagonal grid, 6 neighbors
- MSI: 60 μm pixel size, Cartesian grid, 8 neighbors
- RNA needs 180° rotation to align with MSI

APPROACH:
1. Coordinate-based matching (with rotation + scaling)
2. Resolution-invariant descriptors (for robustness)
3. Spatial GNN Embeddings (GATv2) for structural feature extraction

This version removes the Cross-Modal Transformer to focus on GNN+Descriptor matching.
"""

import numpy as np
import scanpy as sc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATConv, GATv2Conv
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors
from scipy.stats import pearsonr, spearmanr
from scipy.interpolate import griddata
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Optional
import pandas as pd
import os
import warnings
from collections import defaultdict
from dataclasses import dataclass
import pickle
import math

warnings.filterwarnings('ignore')


# =============================================================================
# DATA CONFIGURATION
# =============================================================================

RNA_PIXEL_SIZE = 55  # μm (Visium)
MSI_PIXEL_SIZE = 60  # μm
USE_GATV2 = True  # Use GATv2 (recommended) or GAT (set to False)

MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_4/dummy_data_1/msi/"
MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common.h5ad", "halfbrain_yc_2_filtered_common.h5ad",
    "halfbrain_yc_3_filtered_common.h5ad", "halfbrain_yc_4_filtered_common.h5ad",
    "halfbrain_yad_1_filtered_common.h5ad", "halfbrain_yad_2_filtered_common.h5ad",
    "halfbrain_yad_3_filtered_common.h5ad", "halfbrain_yad_4_filtered_common.h5ad",
    "halfbrain_ac_1_filtered_common.h5ad", "halfbrain_ac_2_filtered_common.h5ad",
    "halfbrain_ac_3_filtered_common.h5ad", "halfbrain_ac_4_filtered_common.h5ad",
    "halfbrain_aad_1_filtered_common.h5ad", "halfbrain_aad_2_filtered_common.h5ad",
    "halfbrain_aad_3_filtered_common.h5ad", "halfbrain_aad_4_filtered_common.h5ad"
]
MSI_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

RNA_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_4/dummy_data_1/rna/"
RNA_SAMPLE_FILES = [
    "YC_1.h5ad", "YC_2.h5ad", "YC_3.h5ad", "YC_4.h5ad",
    "YAD_1.h5ad", "YAD_2.h5ad", "YAD_3.h5ad", "YAD_4.h5ad",
    "AC_1.h5ad", "AC_2.h5ad", "AC_3.h5ad", "AC_4.h5ad",
    "AAD_1.h5ad", "AAD_2.h5ad", "AAD_3.h5ad", "AAD_4.h5ad"
]
RNA_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

AAD_TARGET_GENES = ['Gene_Gradient_X', 'Gene_Radial_Center', 'Gene_Blob_TopRight', 'Gene_Stripes_Vertical']



In [ ]:

# =============================================================================
# COORDINATE TRANSFORMATION
# =============================================================================

def rotate_180(coords: np.ndarray) -> np.ndarray:
    """Rotate coordinates 180° around centroid"""
    center = coords.mean(axis=0)
    return 2 * center - coords


def align_rna_to_msi(rna_coords: np.ndarray, msi_coords: np.ndarray) -> np.ndarray:
    """Align RNA coordinates to MSI coordinate system with 180° rotation and scaling"""
    rotated = rotate_180(rna_coords)
    
    rna_min, rna_max = rotated.min(axis=0), rotated.max(axis=0)
    msi_min, msi_max = msi_coords.min(axis=0), msi_coords.max(axis=0)
    
    rna_range = rna_max - rna_min
    msi_range = msi_max - msi_min
    scale = msi_range / (rna_range + 1e-8)
    
    return (rotated - rna_min) * scale + msi_min


# =============================================================================
# RESOLUTION-INVARIANT DESCRIPTORS
# =============================================================================

def compute_value_histogram(values: np.ndarray, n_bins: int = 50) -> np.ndarray:
    """Value distribution histogram"""
    hist, _ = np.histogram(values, bins=n_bins, density=True)
    return hist / (hist.sum() + 1e-8)


def compute_spatial_histogram(coords: np.ndarray, values: np.ndarray, n_bins: int = 10) -> np.ndarray:
    """2D spatial histogram (normalized coordinates)"""
    norm = (coords - coords.min(axis=0)) / (coords.max(axis=0) - coords.min(axis=0) + 1e-8)
    
    hist = np.zeros((n_bins, n_bins))
    counts = np.zeros((n_bins, n_bins))
    
    for i in range(len(coords)):
        x_bin = min(int(norm[i, 0] * n_bins), n_bins - 1)
        y_bin = min(int(norm[i, 1] * n_bins), n_bins - 1)
        hist[y_bin, x_bin] += values[i]
        counts[y_bin, x_bin] += 1
    
    hist = np.divide(hist, counts, where=counts > 0, out=np.zeros_like(hist))
    
    if hist.max() > hist.min():
        hist = (hist - hist.min()) / (hist.max() - hist.min())
    
    return hist


def compute_radial_profile(coords: np.ndarray, values: np.ndarray, n_rings: int = 10) -> np.ndarray:
    """Radial distribution from centroid"""
    norm = (coords - coords.min(axis=0)) / (coords.max(axis=0) - coords.min(axis=0) + 1e-8)
    centroid = norm.mean(axis=0)
    
    distances = np.linalg.norm(norm - centroid, axis=1)
    max_dist = distances.max()
    
    profile = np.zeros(n_rings)
    counts = np.zeros(n_rings)
    
    for i in range(len(coords)):
        ring = min(int(distances[i] / max_dist * n_rings), n_rings - 1)
        profile[ring] += values[i]
        counts[ring] += 1
    
    profile = np.divide(profile, counts, where=counts > 0, out=np.zeros_like(profile))
    
    if profile.max() > profile.min():
        profile = (profile - profile.min()) / (profile.max() - profile.min())
    
    return profile


def compute_quadrant_stats(coords: np.ndarray, values: np.ndarray, n_div: int = 3) -> np.ndarray:
    """Statistics in spatial quadrants"""
    norm = (coords - coords.min(axis=0)) / (coords.max(axis=0) - coords.min(axis=0) + 1e-8)
    
    stats = []
    for qx in range(n_div):
        for qy in range(n_div):
            x_min, x_max = qx / n_div, (qx + 1) / n_div
            y_min, y_max = qy / n_div, (qy + 1) / n_div
            
            mask = (
                (norm[:, 0] >= x_min) & (norm[:, 0] < x_max) &
                (norm[:, 1] >= y_min) & (norm[:, 1] < y_max)
            )
            
            if mask.sum() > 0:
                q_vals = values[mask]
                stats.extend([np.mean(q_vals), np.std(q_vals)])
            else:
                stats.extend([0, 0])
    
    stats = np.array(stats)
    if stats.max() > stats.min():
        stats = (stats - stats.min()) / (stats.max() - stats.min())
    
    return stats


def compute_morans_i(coords: np.ndarray, values: np.ndarray, k: int = 8) -> float:
    """Moran's I spatial autocorrelation"""
    norm = (coords - coords.min(axis=0)) / (coords.max(axis=0) - coords.min(axis=0) + 1e-8)
    
    nn = NearestNeighbors(n_neighbors=k + 1)
    nn.fit(norm)
    _, indices = nn.kneighbors(norm)
    
    n = len(values)
    mean_val = values.mean()
    denom = np.sum((values - mean_val) ** 2)
    
    if denom == 0:
        return 0
    
    numer = 0
    w_sum = 0
    
    for i in range(n):
        for j in indices[i, 1:]:
            numer += (values[i] - mean_val) * (values[j] - mean_val)
            w_sum += 1
    
    return (n / w_sum) * (numer / denom) if w_sum > 0 else 0


# =============================================================================
# SPATIAL SIGNATURE (STRIPPED)
# =============================================================================

@dataclass
class SpatialSignature:
    """Complete spatial signature with GNN embeddings and Descriptors"""
    sample_id: str
    feature_name: str
    feature_type: str
    
    # GNN embeddings
    global_embedding: np.ndarray
    node_embeddings: np.ndarray
    node_importance: np.ndarray
    
    # Descriptors
    value_histogram: np.ndarray = None
    spatial_histogram: np.ndarray = None
    radial_profile: np.ndarray = None
    quadrant_stats: np.ndarray = None
    morans_i: float = 0.0
    
    # Raw data
    coordinates: np.ndarray = None
    raw_values: np.ndarray = None
    
    # Aligned coordinates
    aligned_coordinates: Optional[np.ndarray] = None


# =============================================================================
# GNN MODEL
# =============================================================================

class SpatialGNN(nn.Module):
    """
    Spatial GNN for embeddings using GATv2.
    """
    
    def __init__(
        self,
        input_dim: int = None,
        hidden_dim: int = 128,
        embedding_dim: int = 64,
        num_heads: int = 4,
        num_layers: int = 3,
        projection_dim: int = 64,
        use_gatv2: bool = None 
    ):
        super().__init__()
        
        self.projection_dim = projection_dim
        self.use_gatv2 = use_gatv2 if use_gatv2 is not None else USE_GATV2
        self.input_projections = nn.ModuleDict()
        
        if input_dim is not None:
            self.input_projections[str(input_dim)] = nn.Sequential(
                nn.Linear(input_dim, projection_dim),
                nn.LayerNorm(projection_dim),
                nn.GELU()
            )
        
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        
        # Choose between GAT and GATv2
        ConvClass = GATv2Conv if self.use_gatv2 else GATConv
        
        self.convs.append(ConvClass(projection_dim, hidden_dim // num_heads,
                                    heads=num_heads, concat=True, dropout=0.1))
        self.norms.append(nn.LayerNorm(hidden_dim))
        
        for _ in range(num_layers - 2):
            self.convs.append(ConvClass(hidden_dim, hidden_dim // num_heads,
                                        heads=num_heads, concat=True, dropout=0.1))
            self.norms.append(nn.LayerNorm(hidden_dim))
        
        self.convs.append(ConvClass(hidden_dim, embedding_dim, heads=1, concat=False, dropout=0.1))
        self.norms.append(nn.LayerNorm(embedding_dim))
        
        self.importance_head = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
        
        self.pool_attention = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, projection_dim)
        )
    
    def _get_projection(self, dim):
        key = str(dim)
        if key not in self.input_projections:
            device = next(self.parameters()).device
            self.input_projections[key] = nn.Sequential(
                nn.Linear(dim, self.projection_dim),
                nn.LayerNorm(self.projection_dim),
                nn.GELU()
            ).to(device)
        return self.input_projections[key]
    
    def forward(self, x, edge_index, bio_importance=None):
        h_input = self._get_projection(x.size(1))(x)
        
        h = h_input
        for conv, norm in zip(self.convs, self.norms):
            h = conv(h, edge_index)
            h = norm(h)
            h = F.gelu(h)
        
        node_embeddings = h
        
        learned_imp = self.importance_head(node_embeddings).squeeze(-1)
        if bio_importance is not None:
            node_importance = 0.5 * learned_imp + 0.5 * bio_importance
        else:
            node_importance = learned_imp
        node_importance = node_importance / (node_importance.max() + 1e-8)
        
        weight = node_importance / (node_importance.sum() + 1e-8) * x.size(0)
        weighted_emb = node_embeddings * weight.unsqueeze(-1)
        
        attn = F.softmax(self.pool_attention(node_embeddings).squeeze(-1), dim=0)
        global_emb = (attn.unsqueeze(-1) * weighted_emb).sum(dim=0)
        
        reconstructed = self.decoder(node_embeddings)
        
        return {
            'node_embeddings': node_embeddings,
            'global_embedding': global_emb,
            'node_importance': node_importance,
            'reconstructed': reconstructed,
            'h_input': h_input
        }


class SpatialLoss(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, output, edge_index):
        recon = F.mse_loss(output['reconstructed'], output['h_input'])
        src, dst = edge_index[0], edge_index[1]
        smooth = ((output['node_embeddings'][src] - output['node_embeddings'][dst]) ** 2).mean()
        return recon + 0.3 * smooth


# =============================================================================
# SIMILARITY COMPUTATION (MODIFIED)
# =============================================================================

def compute_coordinate_based_similarity(gene_sig: SpatialSignature, mz_sig: SpatialSignature, grid_res: int = 50) -> dict:
    """Compute similarity using aligned coordinates"""
    gene_coords = gene_sig.aligned_coordinates if gene_sig.aligned_coordinates is not None else gene_sig.coordinates
    mz_coords = mz_sig.coordinates
    
    x_min = min(gene_coords[:, 0].min(), mz_coords[:, 0].min())
    x_max = max(gene_coords[:, 0].max(), mz_coords[:, 0].max())
    y_min = min(gene_coords[:, 1].min(), mz_coords[:, 1].min())
    y_max = max(gene_coords[:, 1].max(), mz_coords[:, 1].max())
    
    grid_x, grid_y = np.meshgrid(
        np.linspace(x_min, x_max, grid_res),
        np.linspace(y_min, y_max, grid_res)
    )
    
    gene_grid = griddata(gene_coords, gene_sig.raw_values, (grid_x, grid_y), method='linear')
    mz_grid = griddata(mz_coords, mz_sig.raw_values, (grid_x, grid_y), method='linear')
    
    gene_imp_grid = griddata(gene_coords, gene_sig.node_importance, (grid_x, grid_y), method='linear')
    mz_imp_grid = griddata(mz_coords, mz_sig.node_importance, (grid_x, grid_y), method='linear')
    
    mask = ~(np.isnan(gene_grid) | np.isnan(mz_grid))
    if mask.sum() > 10:
        value_corr, _ = pearsonr(gene_grid[mask], mz_grid[mask])
        value_corr = value_corr if not np.isnan(value_corr) else 0
    else:
        value_corr = 0
    
    mask_imp = ~(np.isnan(gene_imp_grid) | np.isnan(mz_imp_grid))
    if mask_imp.sum() > 0:
        g_imp = gene_imp_grid.copy()
        m_imp = mz_imp_grid.copy()
        g_imp[np.isnan(g_imp)] = 0
        m_imp[np.isnan(m_imp)] = 0
        
        g_imp = g_imp / (g_imp.max() + 1e-8)
        m_imp = m_imp / (m_imp.max() + 1e-8)
        
        intersection = np.minimum(g_imp, m_imp).sum()
        union = np.maximum(g_imp, m_imp).sum()
        importance_iou = intersection / (union + 1e-8)
        
        imp_corr, _ = pearsonr(g_imp[mask_imp], m_imp[mask_imp])
        imp_corr = imp_corr if not np.isnan(imp_corr) else 0
    else:
        importance_iou = 0
        imp_corr = 0
    
    return {
        'value_correlation': value_corr,
        'importance_iou': importance_iou,
        'importance_correlation': imp_corr
    }


def compute_descriptor_similarity(gene_sig: SpatialSignature, mz_sig: SpatialSignature) -> dict:
    """Compute similarity using resolution-invariant descriptors"""
    
    g_emb = gene_sig.global_embedding.flatten()
    m_emb = mz_sig.global_embedding.flatten()
    emb_cosine = np.dot(g_emb, m_emb) / (np.linalg.norm(g_emb) * np.linalg.norm(m_emb) + 1e-8)
    
    if gene_sig.value_histogram.std() > 0 and mz_sig.value_histogram.std() > 0:
        val_hist_corr, _ = pearsonr(gene_sig.value_histogram, mz_sig.value_histogram)
        val_hist_corr = val_hist_corr if not np.isnan(val_hist_corr) else 0
    else:
        val_hist_corr = 0
    
    g_spatial = gene_sig.spatial_histogram.flatten()
    m_spatial = mz_sig.spatial_histogram.flatten()
    if g_spatial.std() > 0 and m_spatial.std() > 0:
        spatial_hist_corr, _ = pearsonr(g_spatial, m_spatial)
        spatial_hist_corr = spatial_hist_corr if not np.isnan(spatial_hist_corr) else 0
    else:
        spatial_hist_corr = 0
    
    if gene_sig.radial_profile.std() > 0 and mz_sig.radial_profile.std() > 0:
        radial_corr, _ = pearsonr(gene_sig.radial_profile, mz_sig.radial_profile)
        radial_corr = radial_corr if not np.isnan(radial_corr) else 0
    else:
        radial_corr = 0
    
    if gene_sig.quadrant_stats.std() > 0 and mz_sig.quadrant_stats.std() > 0:
        quad_corr, _ = pearsonr(gene_sig.quadrant_stats, mz_sig.quadrant_stats)
        quad_corr = quad_corr if not np.isnan(quad_corr) else 0
    else:
        quad_corr = 0
    
    morans_diff = abs(gene_sig.morans_i - mz_sig.morans_i)
    morans_sim = 1 / (1 + morans_diff)
    
    return {
        'embedding_cosine': emb_cosine,
        'value_hist_corr': val_hist_corr,
        'spatial_hist_corr': spatial_hist_corr,
        'radial_corr': radial_corr,
        'quadrant_corr': quad_corr,
        'morans_similarity': morans_sim
    }


def compute_combined_score(coord_sim: dict, desc_sim: dict) -> float:
    """
    Combined score using coordinate-based and descriptor-based metrics.
    Weights normalized to sum to 1.0 (50% Coord, 50% Descriptor).
    """
    # Normalizing factor (originally 0.7 total weight, now we scale up to 1.0)
    scale = 1.0 / 0.7
    
    # Coordinate-based (Weighted ~50%)
    coord_score = (
        0.15 * max(coord_sim['value_correlation'], 0) +
        0.10 * coord_sim['importance_iou'] +
        0.10 * max(coord_sim['importance_correlation'], 0)
    ) * scale
    
    # Descriptor-based (Weighted ~50%)
    desc_score = (
        0.10 * desc_sim['embedding_cosine'] +
        0.08 * max(desc_sim['spatial_hist_corr'], 0) +
        0.08 * max(desc_sim['radial_corr'], 0) +
        0.04 * max(desc_sim['quadrant_corr'], 0) +
        0.03 * desc_sim['morans_similarity'] +
        0.02 * max(desc_sim['value_hist_corr'], 0)
    ) * scale
    
    return coord_score + desc_score


# =============================================================================
# MAIN MATCHER
# =============================================================================

class SpatialPatternMatcher:
    """Matcher using Spatial GNN and Descriptors (No Transformer)"""
    
    def __init__(
        self,
        output_dir: str = './gene_to_mz_results_v12_notrans',
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    ):
        self.device = device
        self.output_dir = output_dir
        
        subdirs = ['saliency_maps', 'gene_visualizations', 'match_visualizations',
                   'embeddings', 'training_curves', 'descriptors']
        for subdir in subdirs:
            os.makedirs(os.path.join(output_dir, subdir), exist_ok=True)
        
        self.rna_data = {}
        self.msi_data = {}
        self.rna_model = None
        self.msi_model = None
    
    def load_all_data(self):
        """Load data"""
        print("Loading RNA-seq data...")
        print(f"  Pixel size: {RNA_PIXEL_SIZE} μm (hexagonal)")
        for file, sample_id in zip(RNA_SAMPLE_FILES, RNA_SAMPLE_IDS):
            path = os.path.join(RNA_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.rna_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.rna_data[sample_id].shape}")
        
        print(f"\nLoading MSI data...")
        print(f"  Pixel size: {MSI_PIXEL_SIZE} μm (Cartesian)")
        for file, sample_id in zip(MSI_SAMPLE_FILES, MSI_SAMPLE_IDS):
            path = os.path.join(MSI_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.msi_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.msi_data[sample_id].shape}")
    
    def build_graph(self, coords, n_neighbors):
        """Build spatial graph"""
        nn = NearestNeighbors(n_neighbors=n_neighbors + 1)
        nn.fit(coords)
        distances, indices = nn.kneighbors(coords)
        
        median_dist = np.median(distances[:, 1])
        max_dist = median_dist * 1.5
        
        edges = set()
        for i in range(len(coords)):
            for j_idx in range(1, n_neighbors + 1):
                if distances[i, j_idx] <= max_dist:
                    j = indices[i, j_idx]
                    edges.add((i, j))
                    edges.add((j, i))
        
        return torch.tensor(list(edges), dtype=torch.long).t().contiguous()
    
    def compute_bio_importance(self, coords, values, k):
        """Biological importance"""
        nn = NearestNeighbors(n_neighbors=k + 1)
        nn.fit(coords)
        _, indices = nn.kneighbors(coords)
        
        local_var = np.array([np.var(values[indices[i, 1:]]) for i in range(len(coords))])
        local_var = (local_var - local_var.min()) / (local_var.max() - local_var.min() + 1e-8)
        
        val_norm = (values - values.min()) / (values.max() - values.min() + 1e-8)
        
        return 0.5 * local_var + 0.5 * val_norm
    
    def prepare_data(self, coords, features, n_neighbors):
        """Prepare graph data"""
        if features.ndim == 1:
            features = features.reshape(-1, 1)
        
        bio_imp = np.zeros(len(coords))
        for col in range(features.shape[1]):
            bio_imp += self.compute_bio_importance(coords, features[:, col], n_neighbors)
        bio_imp /= features.shape[1]
        
        scaler = RobustScaler()
        features_scaled = scaler.fit_transform(features)
        
        edge_index = self.build_graph(coords, n_neighbors)
        
        return Data(x=torch.tensor(features_scaled, dtype=torch.float32), edge_index=edge_index), bio_imp
    
    def train_gnn_model(self, data_dict, bio_dict, model_type, epochs=150, val_split=0.2):
        """Train GNN model with validation tracking"""
        print(f"\nTraining {model_type.upper()} GNN model...")
        print(f"  Architecture: {'GATv2' if USE_GATV2 else 'GAT'}")
        
        # Split into train/val
        all_samples = list(data_dict.keys())
        n_val = max(1, int(len(all_samples) * val_split))
        val_samples = all_samples[-n_val:]
        train_samples = all_samples[:-n_val]
        
        print(f"  Train samples: {len(train_samples)}, Val samples: {len(val_samples)}")
        
        first_data = list(data_dict.values())[0]
        model = SpatialGNN(input_dim=first_data.x.size(1)).to(self.device)
        
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)
        loss_fn = SpatialLoss()
        
        train_losses = []
        val_losses = []
        best_val_loss = float('inf')
        best_model_state = None
        
        for epoch in range(epochs):
            # Training
            model.train()
            train_loss = 0
            for sample_id in train_samples:
                data = data_dict[sample_id].to(self.device)
                bio_imp = torch.tensor(bio_dict[sample_id], dtype=torch.float32, device=self.device)
                
                optimizer.zero_grad()
                output = model(data.x, data.edge_index, bio_imp)
                loss = loss_fn(output, data.edge_index)
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item()
            
            train_losses.append(train_loss / len(train_samples))
            
            # Validation
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for sample_id in val_samples:
                    data = data_dict[sample_id].to(self.device)
                    bio_imp = torch.tensor(bio_dict[sample_id], dtype=torch.float32, device=self.device)
                    
                    output = model(data.x, data.edge_index, bio_imp)
                    loss = loss_fn(output, data.edge_index)
                    val_loss += loss.item()
            
            val_losses.append(val_loss / len(val_samples))
            scheduler.step(val_losses[-1])
            
            # Save best model
            if val_losses[-1] < best_val_loss:
                best_val_loss = val_losses[-1]
                best_model_state = model.state_dict().copy()
            
            if (epoch + 1) % 30 == 0:
                print(f"  Epoch {epoch+1}/{epochs}, Train: {train_losses[-1]:.4f}, Val: {val_losses[-1]:.4f}")
        
        # Load best model
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
            print(f"  Loaded best model with val loss: {best_val_loss:.4f}")
        
        # Plot train and validation curves
        plt.figure(figsize=(10, 5))
        plt.plot(train_losses, label='Train Loss', color='blue')
        plt.plot(val_losses, label='Validation Loss', color='red', linestyle='--')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title(f'{model_type.upper()} GNN Training')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # Mark best validation point
        best_epoch = np.argmin(val_losses)
        plt.axvline(x=best_epoch, color='green', linestyle=':', alpha=0.7, label=f'Best Val (epoch {best_epoch})')
        plt.scatter([best_epoch], [val_losses[best_epoch]], color='green', s=100, zorder=5)
        
        plt.legend()
        plt.savefig(os.path.join(self.output_dir, 'training_curves', f'{model_type}_gnn_loss.png'), dpi=150)
        plt.close()
        
        return model
    
    def extract_signature(
        self,
        coords: np.ndarray,
        values: np.ndarray,
        sample_id: str,
        feature_name: str,
        feature_type: str,
        model: SpatialGNN,
        n_neighbors: int,
        msi_coords: Optional[np.ndarray] = None
    ) -> SpatialSignature:
        """Extract signature with all descriptors"""
        
        bio_imp = self.compute_bio_importance(coords, values, n_neighbors)
        data, _ = self.prepare_data(coords, values, n_neighbors)
        data = data.to(self.device)
        bio_tensor = torch.tensor(bio_imp, dtype=torch.float32, device=self.device)
        
        model.eval()
        with torch.no_grad():
            output = model(data.x, data.edge_index, bio_tensor)
        
        val_hist = compute_value_histogram(values)
        spatial_hist = compute_spatial_histogram(coords, values)
        radial = compute_radial_profile(coords, values)
        quadrant = compute_quadrant_stats(coords, values)
        morans = compute_morans_i(coords, values, n_neighbors)
        
        aligned = None
        if feature_type == 'gene' and msi_coords is not None:
            aligned = align_rna_to_msi(coords, msi_coords)
        
        return SpatialSignature(
            sample_id=sample_id,
            feature_name=feature_name,
            feature_type=feature_type,
            global_embedding=output['global_embedding'].cpu().numpy(),
            node_embeddings=output['node_embeddings'].cpu().numpy(),
            node_importance=output['node_importance'].cpu().numpy(),
            value_histogram=val_hist,
            spatial_histogram=spatial_hist,
            radial_profile=radial,
            quadrant_stats=quadrant,
            morans_i=morans,
            coordinates=coords,
            raw_values=values,
            aligned_coordinates=aligned
        )
    
    def visualize_saliency_map(self, sig: SpatialSignature, save_path: str):
        """Visualize saliency/importance map"""
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        
        # Use aligned coordinates if available (for genes)
        coords = sig.aligned_coordinates if sig.aligned_coordinates is not None else sig.coordinates
        
        # Row 0: Importance maps
        im1 = axes[0, 0].scatter(coords[:, 0], coords[:, 1],
                                c=sig.node_importance, cmap='hot', s=15, alpha=0.8)
        axes[0, 0].set_title(f'{sig.feature_name} - Full Importance Map', fontweight='bold')
        axes[0, 0].set_xlabel('X (μm)')
        axes[0, 0].set_ylabel('Y (μm)')
        plt.colorbar(im1, ax=axes[0, 0], label='Importance')
        
        # Top 10%
        thresh_90 = np.percentile(sig.node_importance, 90)
        mask_90 = sig.node_importance >= thresh_90
        axes[0, 1].scatter(coords[:, 0], coords[:, 1], c='lightgray', s=5, alpha=0.3)
        if mask_90.sum() > 0:
            im2 = axes[0, 1].scatter(coords[mask_90, 0], coords[mask_90, 1],
                                      c=sig.node_importance[mask_90], cmap='hot', s=30)
            plt.colorbar(im2, ax=axes[0, 1], label='Importance')
        axes[0, 1].set_title(f'Top 10% Salient Regions ({mask_90.sum()} spots)', fontweight='bold')
        axes[0, 1].set_xlabel('X (μm)')
        axes[0, 1].set_ylabel('Y (μm)')
        
        # Top 5%
        thresh_95 = np.percentile(sig.node_importance, 95)
        mask_95 = sig.node_importance >= thresh_95
        axes[0, 2].scatter(coords[:, 0], coords[:, 1], c='lightgray', s=5, alpha=0.3)
        if mask_95.sum() > 0:
            im3 = axes[0, 2].scatter(coords[mask_95, 0], coords[mask_95, 1],
                                      c=sig.node_importance[mask_95], cmap='hot', s=50)
            plt.colorbar(im3, ax=axes[0, 2], label='Importance')
        axes[0, 2].set_title(f'Top 5% Salient Regions ({mask_95.sum()} spots)', fontweight='bold')
        axes[0, 2].set_xlabel('X (μm)')
        axes[0, 2].set_ylabel('Y (μm)')
        
        # Row 1: Weighted expr
        weighted_expr = sig.raw_values * sig.node_importance
        im4 = axes[1, 0].scatter(coords[:, 0], coords[:, 1],
                                  c=weighted_expr, cmap='viridis', s=15, alpha=0.8)
        axes[1, 0].set_title('Importance-Weighted Expression', fontweight='bold')
        axes[1, 0].set_xlabel('X (μm)')
        axes[1, 0].set_ylabel('Y (μm)')
        plt.colorbar(im4, ax=axes[1, 0], label='Weighted Value')
        
        # Hist
        axes[1, 1].hist(sig.node_importance, bins=50, color='coral', edgecolor='darkred', alpha=0.7)
        axes[1, 1].axvline(thresh_90, color='blue', linestyle='--', label=f'Top 10% ({thresh_90:.3f})')
        axes[1, 1].axvline(thresh_95, color='green', linestyle='--', label=f'Top 5% ({thresh_95:.3f})')
        axes[1, 1].set_title('Importance Score Distribution', fontweight='bold')
        axes[1, 1].set_xlabel('Importance Score')
        axes[1, 1].set_ylabel('Count')
        axes[1, 1].legend()
        
        # Stats
        stats_text = f"""
SALIENCY STATISTICS
═══════════════════════════════

Feature: {sig.feature_name}
Sample:  {sig.sample_id}
Type:    {sig.feature_type.upper()}

Total Spots:     {len(sig.node_importance)}
Top 10% Spots:   {mask_90.sum()}
Top 5% Spots:    {mask_95.sum()}

Importance Scores:
  Mean:   {sig.node_importance.mean():.4f}
  Std:    {sig.node_importance.std():.4f}
  Min:    {sig.node_importance.min():.4f}
  Max:    {sig.node_importance.max():.4f}

Moran's I: {sig.morans_i:.4f}
(Spatial autocorrelation)
        """
        axes[1, 2].text(0.05, 0.95, stats_text, transform=axes[1, 2].transAxes,
                       fontsize=10, verticalalignment='top', family='monospace',
                       bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
        axes[1, 2].axis('off')
        
        plt.suptitle(f'SALIENCY MAP: {sig.feature_name} ({sig.sample_id})',
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        plt.close()
    
    def visualize_signature(self, sig: SpatialSignature, save_path: str):
        """Visualize gene/mz expression patterns and spatial descriptors"""
        fig, axes = plt.subplots(2, 4, figsize=(20, 10))
        
        # Row 0: Expression patterns
        im1 = axes[0, 0].scatter(sig.coordinates[:, 0], sig.coordinates[:, 1],
                                   c=sig.raw_values, cmap='viridis', s=10)
        axes[0, 0].set_title(f'{sig.feature_name} - Original', fontweight='bold')
        axes[0, 0].set_xlabel('X (μm)')
        axes[0, 0].set_ylabel('Y (μm)')
        plt.colorbar(im1, ax=axes[0, 0], label='Expression')
        
        # Aligned coordinates (for genes) or log-transformed
        if sig.aligned_coordinates is not None:
            im2 = axes[0, 1].scatter(sig.aligned_coordinates[:, 0], sig.aligned_coordinates[:, 1],
                                      c=sig.raw_values, cmap='viridis', s=10)
            axes[0, 1].set_title('Aligned (180° rotated)', fontweight='bold')
            plt.colorbar(im2, ax=axes[0, 1], label='Expression')
        else:
            log_vals = np.log1p(sig.raw_values)
            im2 = axes[0, 1].scatter(sig.coordinates[:, 0], sig.coordinates[:, 1],
                                      c=log_vals, cmap='viridis', s=10)
            axes[0, 1].set_title('Log-transformed', fontweight='bold')
            plt.colorbar(im2, ax=axes[0, 1], label='log(1+x)')
        axes[0, 1].set_xlabel('X (μm)')
        axes[0, 1].set_ylabel('Y (μm)')
        
        # Spatial histogram
        im3 = axes[0, 2].imshow(sig.spatial_histogram, cmap='viridis', origin='lower',
                                 aspect='auto')
        axes[0, 2].set_title('Spatial Histogram (10x10)', fontweight='bold')
        axes[0, 2].set_xlabel('X bin')
        axes[0, 2].set_ylabel('Y bin')
        plt.colorbar(im3, ax=axes[0, 2], label='Mean Expression')
        
        # Radial profile
        axes[0, 3].plot(sig.radial_profile, 'b-', linewidth=2, marker='o', markersize=4)
        axes[0, 3].fill_between(range(len(sig.radial_profile)), sig.radial_profile, alpha=0.3)
        axes[0, 3].set_title('Radial Profile (from centroid)', fontweight='bold')
        axes[0, 3].set_xlabel('Ring (center → edge)')
        axes[0, 3].set_ylabel('Normalized Expression')
        axes[0, 3].grid(True, alpha=0.3)
        
        # Row 1: Descriptors
        axes[1, 0].bar(range(len(sig.value_histogram)), sig.value_histogram, 
                       color='steelblue', edgecolor='darkblue', alpha=0.7)
        axes[1, 0].set_title('Expression Distribution', fontweight='bold')
        axes[1, 0].set_xlabel('Bin')
        axes[1, 0].set_ylabel('Density')
        
        # Quadrant stats
        n_div = 3
        quad_means = sig.quadrant_stats[::2].reshape(n_div, n_div)
        im4 = axes[1, 1].imshow(quad_means, cmap='YlOrRd', origin='lower', aspect='auto')
        axes[1, 1].set_title('Quadrant Mean Expression', fontweight='bold')
        axes[1, 1].set_xlabel('X quadrant')
        axes[1, 1].set_ylabel('Y quadrant')
        for i in range(n_div):
            for j in range(n_div):
                axes[1, 1].text(j, i, f'{quad_means[i, j]:.2f}', ha='center', va='center', fontsize=9)
        plt.colorbar(im4, ax=axes[1, 1])
        
        # Percentiles
        coords = sig.aligned_coordinates if sig.aligned_coordinates is not None else sig.coordinates
        percentiles = [25, 50, 75, 90]
        colors = ['lightblue', 'steelblue', 'orange', 'red']
        axes[1, 2].scatter(coords[:, 0], coords[:, 1], c='lightgray', s=3, alpha=0.3)
        for pct, color in zip(percentiles, colors):
            thresh = np.percentile(sig.raw_values, pct)
            mask = sig.raw_values >= thresh
            axes[1, 2].scatter(coords[mask, 0], coords[mask, 1], 
                              c=color, s=5, alpha=0.5, label=f'≥{pct}th pct')
        axes[1, 2].set_title('Expression Percentiles', fontweight='bold')
        axes[1, 2].set_xlabel('X (μm)')
        axes[1, 2].set_ylabel('Y (μm)')
        axes[1, 2].legend(loc='upper right', fontsize=8)
        
        # Stats
        stats_text = f"""
EXPRESSION STATISTICS
═══════════════════════════════

Feature: {sig.feature_name}
Sample:  {sig.sample_id}
Type:    {sig.feature_type.upper()}

Expression Values:
  Mean:   {sig.raw_values.mean():.4f}
  Std:    {sig.raw_values.std():.4f}
  Min:    {sig.raw_values.min():.4f}
  Max:    {sig.raw_values.max():.4f}
  Median: {np.median(sig.raw_values):.4f}

Spatial Metrics:
  Moran's I:     {sig.morans_i:.4f}
  Total Spots:   {len(sig.raw_values)}
  Non-zero:      {(sig.raw_values > 0).sum()}

Embedding Dim: {len(sig.global_embedding)}
        """
        axes[1, 3].text(0.05, 0.95, stats_text, transform=axes[1, 3].transAxes,
                       fontsize=10, verticalalignment='top', family='monospace',
                       bbox=dict(boxstyle='round', facecolor='lightcyan', alpha=0.8))
        axes[1, 3].axis('off')
        
        plt.suptitle(f'EXPRESSION PATTERN: {sig.feature_name} ({sig.sample_id}) | Moran\'s I: {sig.morans_i:.3f}',
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        plt.close()
    
    def visualize_match(self, gene_sig, mz_sig, coord_sim, desc_sim, save_path):
        """Visualize match (No transformer attention)"""
        combined = compute_combined_score(coord_sim, desc_sim)
        
        fig, axes = plt.subplots(3, 4, figsize=(20, 15))
        
        # Row 0: Gene
        im1 = axes[0, 0].scatter(gene_sig.coordinates[:, 0], gene_sig.coordinates[:, 1],
                                  c=gene_sig.raw_values, cmap='viridis', s=15)
        axes[0, 0].set_title(f'Gene: {gene_sig.feature_name} (Original)', fontweight='bold')
        plt.colorbar(im1, ax=axes[0, 0])
        
        if gene_sig.aligned_coordinates is not None:
            im2 = axes[0, 1].scatter(gene_sig.aligned_coordinates[:, 0], gene_sig.aligned_coordinates[:, 1],
                                      c=gene_sig.raw_values, cmap='viridis', s=15)
            axes[0, 1].set_title('Gene (180° Aligned)', fontweight='bold')
            plt.colorbar(im2, ax=axes[0, 1])
        
        im3 = axes[0, 2].imshow(gene_sig.spatial_histogram, cmap='viridis', origin='lower')
        axes[0, 2].set_title('Gene Spatial Hist', fontweight='bold')
        plt.colorbar(im3, ax=axes[0, 2])
        
        axes[0, 3].plot(gene_sig.radial_profile, 'b-', linewidth=2, label='Gene')
        axes[0, 3].plot(mz_sig.radial_profile, 'r--', linewidth=2, label='m/z')
        axes[0, 3].set_title(f'Radial (r={desc_sim["radial_corr"]:.3f})', fontweight='bold')
        axes[0, 3].legend()
        
        # Row 1: m/z
        im4 = axes[1, 0].scatter(mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
                                  c=mz_sig.raw_values, cmap='viridis', s=5)
        axes[1, 0].set_title(f'm/z: {mz_sig.feature_name}', fontweight='bold')
        plt.colorbar(im4, ax=axes[1, 0])
        
        im5 = axes[1, 1].scatter(mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
                                  c=mz_sig.node_importance, cmap='hot', s=5)
        axes[1, 1].set_title('m/z Importance', fontweight='bold')
        plt.colorbar(im5, ax=axes[1, 1])
        
        im6 = axes[1, 2].imshow(mz_sig.spatial_histogram, cmap='viridis', origin='lower')
        axes[1, 2].set_title('m/z Spatial Hist', fontweight='bold')
        plt.colorbar(im6, ax=axes[1, 2])
        
        diff = gene_sig.spatial_histogram - mz_sig.spatial_histogram
        im7 = axes[1, 3].imshow(diff, cmap='RdBu_r', origin='lower')
        axes[1, 3].set_title(f'Spatial Diff (r={desc_sim["spatial_hist_corr"]:.3f})', fontweight='bold')
        plt.colorbar(im7, ax=axes[1, 3])
        
        # Row 2: Overlays and metrics
        if gene_sig.aligned_coordinates is not None:
            axes[2, 0].scatter(mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
                              c='blue', s=3, alpha=0.3, label='m/z')
            axes[2, 0].scatter(gene_sig.aligned_coordinates[:, 0], gene_sig.aligned_coordinates[:, 1],
                              c='red', s=10, alpha=0.5, label='Gene')
            axes[2, 0].set_title('Overlay (Gene aligned)', fontweight='bold')
            axes[2, 0].legend()
        
        gene_thresh = np.percentile(gene_sig.node_importance, 70)
        mz_thresh = np.percentile(mz_sig.node_importance, 70)
        gene_mask = gene_sig.node_importance >= gene_thresh
        mz_mask = mz_sig.node_importance >= mz_thresh
        
        if gene_sig.aligned_coordinates is not None:
            axes[2, 1].scatter(mz_sig.coordinates[mz_mask, 0], mz_sig.coordinates[mz_mask, 1],
                              c='blue', s=8, alpha=0.5, label='m/z top 30%')
            axes[2, 1].scatter(gene_sig.aligned_coordinates[gene_mask, 0], 
                              gene_sig.aligned_coordinates[gene_mask, 1],
                              c='red', s=15, alpha=0.7, label='Gene top 30%')
            axes[2, 1].set_title('Top 30% Overlay', fontweight='bold')
            axes[2, 1].legend()
        
        axes[2, 2].bar(range(len(gene_sig.value_histogram)), gene_sig.value_histogram, alpha=0.5, label='Gene')
        axes[2, 2].bar(range(len(mz_sig.value_histogram)), mz_sig.value_histogram, alpha=0.5, label='m/z')
        axes[2, 2].set_title(f'Value Hist (r={desc_sim["value_hist_corr"]:.3f})', fontweight='bold')
        axes[2, 2].legend()
        
        # Metrics
        metrics_text = f"""
COMBINED SCORE: {combined:.4f}
═══════════════════════════════════

COORDINATE-BASED (50%):
  Value correlation:    {coord_sim['value_correlation']:.4f}
  Importance IoU:       {coord_sim['importance_iou']:.4f}
  Importance corr:      {coord_sim['importance_correlation']:.4f}

DESCRIPTOR-BASED (50%):
  Embedding cosine:     {desc_sim['embedding_cosine']:.4f}
  Spatial hist corr:    {desc_sim['spatial_hist_corr']:.4f}
  Radial corr:          {desc_sim['radial_corr']:.4f}
  Quadrant corr:        {desc_sim['quadrant_corr']:.4f}
  Moran's I sim:        {desc_sim['morans_similarity']:.4f}

SPATIAL STATS:
  Gene Moran's I:       {gene_sig.morans_i:.4f}
  m/z Moran's I:        {mz_sig.morans_i:.4f}
        """
        axes[2, 3].text(0.05, 0.95, metrics_text, transform=axes[2, 3].transAxes,
                       fontsize=9, verticalalignment='top', family='monospace',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        axes[2, 3].axis('off')
        
        plt.suptitle(f'Match: {gene_sig.feature_name} ↔ m/z {mz_sig.feature_name} | Score: {combined:.3f}',
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        plt.close()
    
    def find_matches(self, gene_sig, mz_sigs, top_k=20):
        """Find matches using coordinate and descriptor similarity"""
        matches = []
        
        for mz_name, mz_sig in mz_sigs.items():
            coord_sim = compute_coordinate_based_similarity(gene_sig, mz_sig)
            desc_sim = compute_descriptor_similarity(gene_sig, mz_sig)
            combined = compute_combined_score(coord_sim, desc_sim)
            
            matches.append({
                'gene': gene_sig.feature_name,
                'rna_sample': gene_sig.sample_id,
                'mz_feature': mz_name,
                'msi_sample': mz_sig.sample_id,
                **coord_sim,
                **desc_sim,
                'combined_score': combined
            })
        
        return pd.DataFrame(matches).sort_values('combined_score', ascending=False).head(top_k)
    
    def run_analysis(self, epochs_gnn=150, top_k=20):
        """Run analysis with GNN and Descriptors only"""
        print("\n" + "="*70)
        print("GENE-TO-M/Z MATCHING V12 (No Transformer)")
        print(f"RNA: {RNA_PIXEL_SIZE}μm (hexagonal) | MSI: {MSI_PIXEL_SIZE}μm (Cartesian)")
        print("Strategy: Coordinate + Descriptor + GNN Embeddings")
        print("="*70)
        
        # Gene availability
        gene_avail = {}
        for gene in AAD_TARGET_GENES:
            gene_avail[gene] = {s: gene in self.rna_data[s].var_names 
                               for s in RNA_SAMPLE_IDS if s in self.rna_data}
        
        print("\nGene availability:")
        for gene in AAD_TARGET_GENES:
            n = sum(gene_avail[gene].values())
            print(f"  {gene}: {n}/{len(RNA_SAMPLE_IDS)}")
        
        # Organize Splits
        groups = ['YC', 'YAD', 'AC', 'AAD']
        train_ids = []
        val_ids = []
        
        available_samples = list(self.rna_data.keys())
        
        print(f"\nOrganizing Split (Train: 12, Val: 4)...")
        for group in groups:
            group_samples = sorted([s for s in available_samples if s.startswith(f"{group}_")])
            if len(group_samples) >= 4:
                train_ids.extend(group_samples[:3])
                val_ids.append(group_samples[3])
                print(f"  {group}: Train={group_samples[:3]}, Val={[group_samples[3]]}")
            else:
                train_ids.extend(group_samples)

        processing_order = train_ids + val_ids
        final_val_split = len(val_ids) / len(processing_order) if processing_order else 0.2
        
        # PHASE 1: TRAIN GNN MODELS
        print("\n" + "-"*70)
        print("PHASE 1: Training GNN Models (Balanced)")
        print("-"*70)
        
        # RNA GNN
        print(f"\nRNA GNN (6 neighbors, {RNA_PIXEL_SIZE}μm)...")
        rna_train, rna_bio = {}, {}
        for sid in processing_order:
            if sid not in self.rna_data: continue
            adata = self.rna_data[sid]
            coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            features = adata.X[:, :50].toarray() if hasattr(adata.X, 'toarray') else adata.X[:, :50]
            data, bio = self.prepare_data(coords, features, 6)
            rna_train[sid] = data
            rna_bio[sid] = bio
            role = "VAL" if sid in val_ids else "TRAIN"
            print(f"  {sid} ({role}): {data.x.shape}")
        
        self.rna_model = self.train_gnn_model(rna_train, rna_bio, 'rna', epochs_gnn, val_split=final_val_split)
        
        # MSI GNN
        print(f"\nMSI GNN (8 neighbors, {MSI_PIXEL_SIZE}μm)...")
        msi_train, msi_bio = {}, {}
        for sid in processing_order:
            if sid not in self.msi_data: continue
            adata = self.msi_data[sid]
            coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            features = adata.X[:, :50].toarray() if hasattr(adata.X, 'toarray') else adata.X[:, :50]
            data, bio = self.prepare_data(coords, features, 8)
            msi_train[sid] = data
            msi_bio[sid] = bio
            role = "VAL" if sid in val_ids else "TRAIN"
            print(f"  {sid} ({role}): {data.x.shape}")
        
        self.msi_model = self.train_gnn_model(msi_train, msi_bio, 'msi', epochs_gnn, val_split=final_val_split)
        
        # PHASE 3: MATCHING (ALL SAMPLES)
        print("\n" + "-"*70)
        print("PHASE 2: Matching (All Samples)")
        print("-"*70)
        
        all_results = []
        all_top5_results = []
        
        for gene in AAD_TARGET_GENES:
            print(f"\n{'='*50}")
            print(f"Gene: {gene}")
            print(f"{'='*50}")
            
            available = [s for s, a in gene_avail[gene].items() if a]
            if not available:
                continue
            
            for rna_sample in available:
                print(f"\n  {rna_sample}")
                
                adata = self.rna_data[rna_sample]
                rna_coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
                gene_idx = list(adata.var_names).index(gene)
                gene_expr = adata.X[:, gene_idx].toarray().flatten() if hasattr(adata.X, 'toarray') else adata.X[:, gene_idx].flatten()
                
                msi_sample = rna_sample
                if msi_sample not in self.msi_data:
                    print(f"    MSI {msi_sample} not found")
                    continue
                
                msi_adata = self.msi_data[msi_sample]
                msi_coords = np.column_stack([msi_adata.obs['x_um'].values, msi_adata.obs['y_um'].values])
                
                gene_sig = self.extract_signature(
                    rna_coords, gene_expr, rna_sample, gene, 'gene',
                    self.rna_model, 6, msi_coords
                )
                
                self.visualize_saliency_map(
                    gene_sig,
                    os.path.join(self.output_dir, 'saliency_maps', f'{gene}_{rna_sample}.png')
                )
                
                self.visualize_signature(
                    gene_sig,
                    os.path.join(self.output_dir, 'gene_visualizations', f'{gene}_{rna_sample}.png')
                )
                
                with open(os.path.join(self.output_dir, 'embeddings', f'{gene}_{rna_sample}.pkl'), 'wb') as f:
                    pickle.dump({
                        'global_embedding': gene_sig.global_embedding,
                        'node_importance': gene_sig.node_importance,
                        'morans_i': gene_sig.morans_i
                    }, f)

                with open(os.path.join(self.output_dir, 'descriptors', f'{gene}_{rna_sample}_descriptors.pkl'), 'wb') as f:
                    pickle.dump({
                        'feature_name': gene_sig.feature_name,
                        'sample_id': gene_sig.sample_id,
                        'feature_type': gene_sig.feature_type,
                        'value_histogram': gene_sig.value_histogram,
                        'spatial_histogram': gene_sig.spatial_histogram,
                        'radial_profile': gene_sig.radial_profile,
                        'quadrant_stats': gene_sig.quadrant_stats,
                        'morans_i': gene_sig.morans_i,
                        'coordinates': gene_sig.coordinates,
                        'aligned_coordinates': gene_sig.aligned_coordinates,
                        'expression_stats': {
                            'mean': float(gene_sig.raw_values.mean()),
                            'std': float(gene_sig.raw_values.std()),
                            'min': float(gene_sig.raw_values.min()),
                            'max': float(gene_sig.raw_values.max()),
                            'median': float(np.median(gene_sig.raw_values)),
                            'n_spots': len(gene_sig.raw_values),
                            'n_nonzero': int((gene_sig.raw_values > 0).sum()),
                            'sparsity': float((gene_sig.raw_values == 0).sum() / len(gene_sig.raw_values))
                        },
                        'importance_stats': {
                            'mean': float(gene_sig.node_importance.mean()),
                            'std': float(gene_sig.node_importance.std()),
                            'top_10pct_threshold': float(np.percentile(gene_sig.node_importance, 90)),
                            'top_5pct_threshold': float(np.percentile(gene_sig.node_importance, 95)),
                        }
                    }, f)                

                print(f"    Matching vs MSI...")
                msi_data = msi_adata.X.toarray() if hasattr(msi_adata.X, 'toarray') else msi_adata.X
                mz_names = list(msi_adata.var_names)
                
                mz_sigs = {}
                for i, mz_name in enumerate(mz_names):
                    mz_sigs[mz_name] = self.extract_signature(
                        msi_coords, msi_data[:, i], msi_sample, mz_name, 'mz',
                        self.msi_model, 8
                    )
                    if (i + 1) % 100 == 0:
                        print(f"      {i+1}/{len(mz_names)}")
                
                matches = self.find_matches(gene_sig, mz_sigs, top_k)
                all_results.append(matches)

                # Store top 5
                top5_matches = matches.head(5).copy()
                top5_matches['gene'] = gene
                top5_matches['rna_sample'] = rna_sample
                top5_matches['rank'] = range(1, len(top5_matches) + 1)
                all_top5_results.append(top5_matches)
                
                if len(matches) > 0:
                    print(f"\n    Top 5:")
                    for idx in range(min(5, len(matches))):
                        m = matches.iloc[idx]
                        print(f"      {idx+1}. m/z {m['mz_feature']}: {m['combined_score']:.3f}")
                    
                    # Visualize top match        
                    top = matches.iloc[0]
                    top_mz = mz_sigs[top['mz_feature']]
                    coord_sim = compute_coordinate_based_similarity(gene_sig, top_mz)
                    desc_sim = compute_descriptor_similarity(gene_sig, top_mz)
                    
                    self.visualize_match(
                        gene_sig, top_mz, coord_sim, desc_sim,
                        os.path.join(self.output_dir, 'match_visualizations', f'{gene}_{rna_sample}_top.png')
                    )

                    # Save descriptors
                    for idx in range(min(5, len(matches))):
                        m = matches.iloc[idx]
                        mz_name = m['mz_feature']
                        mz_sig = mz_sigs[mz_name]
                        mz_filename = mz_name.replace('/', '_').replace('\\', '_')
                        
                        with open(os.path.join(self.output_dir, 'descriptors', 
                                  f'{gene}_{rna_sample}_match{idx+1}_{mz_filename}_descriptors.pkl'), 'wb') as f:
                            pickle.dump({
                                'gene': gene,
                                'gene_sample': rna_sample,
                                'mz_feature': mz_name,
                                'mz_sample': mz_sig.sample_id,
                                'match_rank': idx + 1,
                                'combined_score': float(m['combined_score']),
                                'mz_value_histogram': mz_sig.value_histogram,
                                'mz_spatial_histogram': mz_sig.spatial_histogram,
                                'mz_radial_profile': mz_sig.radial_profile,
                                'mz_quadrant_stats': mz_sig.quadrant_stats,
                                'mz_morans_i': mz_sig.morans_i,
                                'gene_value_histogram': gene_sig.value_histogram,
                                'gene_spatial_histogram': gene_sig.spatial_histogram,
                                'gene_radial_profile': gene_sig.radial_profile,
                                'gene_quadrant_stats': gene_sig.quadrant_stats,
                                'gene_morans_i': gene_sig.morans_i,
                                'similarity_scores': {k: float(v) if isinstance(v, (int, float, np.floating)) else v 
                                                  for k, v in m.items() if k not in ['gene', 'rna_sample', 'mz_feature', 'msi_sample']}
                            }, f)
        
        # Save results
        if all_results:
            results = pd.concat(all_results, ignore_index=True)
            results = results.sort_values('combined_score', ascending=False)
            results.to_csv(os.path.join(self.output_dir, 'gene_to_mz_matches_v12_notrans.csv'), index=False)
             
            if all_top5_results:
                top5_df = pd.concat(all_top5_results, ignore_index=True)
                priority_cols = ['gene', 'rna_sample', 'rank', 'mz_feature', 'msi_sample', 'combined_score']
                other_cols = [c for c in top5_df.columns if c not in priority_cols]
                top5_df = top5_df[priority_cols + other_cols]
                top5_df = top5_df.sort_values(['gene', 'rna_sample', 'rank'])
                top5_df.to_csv(os.path.join(self.output_dir, 'gene_to_mz_top5_matches_all_scores.csv'), index=False)
            
            print(f"\nSaved results to: {self.output_dir}")
            
            print("\n" + "="*70)
            print("TOP MATCHES")
            print("="*70)
            
            for gene in AAD_TARGET_GENES:
                gr = results[results['gene'] == gene]
                if len(gr) > 0:
                    t = gr.iloc[0]
                    print(f"\n{gene}: m/z {t['mz_feature']} ({t['rna_sample']}) = {t['combined_score']:.3f}")
            
            return results
        
        return None


In [ ]:
    
def main():
    print("="*70)
    print("V12: Spatial GNN Matching (No Transformer)")
    print(f"RNA: {RNA_PIXEL_SIZE}μm | MSI: {MSI_PIXEL_SIZE}μm")
    print(f"GNN Architecture: {'GATv2' if USE_GATV2 else 'GAT'}")
    print("="*70)
    
    matcher = SpatialPatternMatcher(output_dir='./test_gene_to_mz_results_v12_notrans')
    matcher.load_all_data()
    results = matcher.run_analysis(epochs_gnn=300, top_k=20)
    
    print("\n" + "="*70)
    print("COMPLETE!")
    print("="*70)
    
    return matcher, results


if __name__ == "__main__":
    matcher, results = main()